In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.optim import Adam
from torchvision.datasets.mnist import MNIST
from torch.utils.data import DataLoader
import numpy as np


In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self, d_model, img_size, patch_size, n_channels):
    super().__init__()

    self.d_model = d_model # Dimensionality of Model
    self.img_size = img_size # Image Size
    self.patch_size = patch_size # Patch Size
    self.n_channels = n_channels # Number of Channels

    self.linear_project = nn.Conv2d(self.n_channels, self.d_model, kernel_size=self.patch_size, stride=self.patch_size)

  def forward(self, x):
    x = self.linear_project(x) # (B, C, H, W) -> (B, d_model, P_col, P_row)

    x = x.flatten(2) # (B, d_model, P_col, P_row) -> (B, d_model, P)

    x = x.transpose(1, 2) # (B, d_model, P) -> (B, P, d_model)
    
    return x

In [1]:
!pip install "stamp[all] @ git+https://github.com/KatherLab/STAMP"


  Cloning https://github.com/KatherLab/STAMP to c:\users\versu\appdata\local\temp\pip-install-a1rai86c\stamp_7e4050a214dd4b89a7220a821f123b05
  Resolved https://github.com/KatherLab/STAMP to commit 0042e6f2630bcfc4351a39a0c6650a195589f850
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached beartype-0.20.2-py3-none-any.whl.metadata (33 kB)
  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached h5py-3.13.0-cp311-cp311-win_amd64.whl.metadata (2.5 kB)
  Using cached jaxtyping-0.3.2-py3-none-any.whl.metadata (7.0 kB)
  Using cached lightning-2.5.1.post0-py3-none-any.whl.metadata (39 kB)
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ------ ------------

  Running command git clone --filter=blob:none --quiet https://github.com/KatherLab/STAMP 'C:\Users\versu\AppData\Local\Temp\pip-install-a1rai86c\stamp_7e4050a214dd4b89a7220a821f123b05'
  Running command git clone --filter=blob:none --quiet https://github.com/Mahmoodlab/CONCH.git 'C:\Users\versu\AppData\Local\Temp\pip-install-a1rai86c\conch_61695236553a4975941fbba4f1db4ac3'
  Running command git rev-parse -q --verify 'sha^02d6ac59cc20874bff0f581de258c2b257f69a84'
  Running command git fetch -q https://github.com/Mahmoodlab/CONCH.git 02d6ac59cc20874bff0f581de258c2b257f69a84
  Running command git checkout -q 02d6ac59cc20874bff0f581de258c2b257f69a84
  Running command git clone --filter=blob:none --quiet https://github.com/KatherLab/uni.git 'C:\Users\versu\AppData\Local\Temp\pip-install-a1rai86c\uni_977d7dfdf5fc4e9bbcc8fef9a8498737'
  Running command git rev-parse -q --verify 'sha^f37c299eb0bffa0e585f120974082cfec6ee6d53'
  Running command git fetch -q https://github.com/KatherLab/uni.git 

In [6]:
from stamp.modeling.vision_transformer import VisionTransformer

def test_vision_transformer_dims(
    # arbitrarily chosen constants
    num_classes: int = 3,
    batch_size: int = 6,
    n_tiles: int = 75,
    input_dim: int = 456,
    n_heads: int = 4,
) -> None:
    model = VisionTransformer(
        dim_output=num_classes,
        dim_input=input_dim,
        dim_model=n_heads * 33,
        n_layers=3,
        n_heads=n_heads,
        dim_feedforward=135,
        dropout=0.12,
        use_alibi=False,
    )

    bags = torch.rand((batch_size, n_tiles, input_dim))
    coords = torch.rand((batch_size, n_tiles, 2))
    mask = torch.rand((batch_size, n_tiles)) > 0.5
    logits = model.forward(bags, coords=coords, mask=mask)
    assert logits.shape == (batch_size, num_classes)
